# Development time split — does the ranking predict, or only describe?

`merchant_ranking_baseline.ipynb` ranks merchants using historical data. That tells us who
*was* commercially valuable; onboarding needs evidence about who performs next.

This notebook is deliberately a **development diagnostic**, not an independent final test.
It is used to understand why v1 succeeds or fails and to choose the hybrid design.

| | Period | Used for |
| --- | --- | --- |
| **Training** | 1 Mar 2021 – 28 Feb 2022 | rebuild features using 12 complete months |
| **Development** | 1 Mar 2022 – 31 Aug 2022 | diagnose v1 and compare candidate weightings |
| **September evaluation** | 1 Sep 2022 – 30 Sep 2022 | history-only comparison; also inspected by supporting experiments |

28 February 2021 is excluded because the dataset begins on that date, so it is not a complete
month. October 2022 is excluded because it is incomplete.

Three development questions:

1. **Does the score track future revenue?** Spearman correlation between the score and
   realised development-window revenue.
2. **Does it beat the obvious baseline?** Compare against ranking by past BNPL revenue.
3. **How much money would the pick have captured?** Compare the predicted top 100 with the
   best possible top 100 in the development window.

Because the hybrid design is informed by this notebook, **March–August must not later be described
as an untouched final test**.


In [1]:
K = 300


# Resolve imports by repository contents, regardless of notebook location.
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scripts" / "process_cleaning.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))


from pathlib import Path

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "tables").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from inside the repository "
        "containing data/tables/."
    )

PROJECT_ROOT = find_project_root()
PROCESSED = PROJECT_ROOT / "data" / "tables" / "processed"

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F

TRAIN_START, TRAIN_END = "2021-03-01", "2022-02-28"
DEV_START, DEV_END = "2022-03-01", "2022-08-31"
FINAL_HOLDOUT_START, FINAL_HOLDOUT_END = "2022-09-01", "2022-09-30"

spark = (
    SparkSession.builder.config("spark.sql.session.timeZone", "Etc/UTC").appName("MAST30034 Project 2 - Development time split")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

transactions = spark.read.parquet(str(PROCESSED / "transactions_clean"))
merchants = spark.read.parquet(str(PROCESSED / "merchants_clean"))
consumers = spark.read.parquet(str(PROCESSED / "consumers_sa2"))

train = transactions.filter(F.col("order_datetime").between(TRAIN_START, TRAIN_END))
dev = transactions.filter(F.col("order_datetime").between(DEV_START, DEV_END))

print(f"training    : {train.count():,} transactions  ({TRAIN_START} to {TRAIN_END})")
print(f"development : {dev.count():,} transactions  ({DEV_START} to {DEV_END})")
print(
    "reserved final holdout: "
    f"{FINAL_HOLDOUT_START} to {FINAL_HOLDOUT_END}"
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 23:25:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


training    : 8,155,190 transactions  (2021-03-01 to 2022-02-28)
development : 4,468,480 transactions  (2022-03-01 to 2022-08-31)
reserved final holdout: 2022-09-01 to 2022-09-30


## 1. Rebuild the production features using training data only

The full `merchant_features` table cannot be used for this experiment because it includes
transactions after February 2022. The features below are therefore rebuilt from the training
window only.

The definitions mirror the production feature engineering as closely as possible:

- 12 complete calendar months are used for activity and stability;
- inactive months are explicitly filled with zero before calculating monthly volatility;
- growth compares the first three complete months (Mar–May 2021) with the last three
  complete months of the training window (Dec 2021–Feb 2022);
- growth is only calculated when the merchant is active in at least two months of both windows.


In [2]:
import pandas as pd
from pyspark.sql import functions as F

def _build_window_features(
    tx,
    months_in_window,
    early_start,
    early_end_exclusive,
    recent_start,
    recent_end_exclusive,
    window_start,
    window_end,
    *, merchants, consumers,
):
    spark = tx.sparkSession
    tx = tx.filter((F.col("order_datetime") >= F.lit(window_start)) &
                   (F.col("order_datetime") < F.date_add(F.to_date(F.lit(window_end)), 1)))
    core = (
        tx.groupBy("merchant_abn")
        .agg(
            F.sum("dollar_value").alias("gmv"),
            F.count("*").alias("transaction_count"),
            F.countDistinct("user_id").alias("unique_customers"),
            F.countDistinct(F.trunc("order_datetime", "month")).alias("active_months"),
        )
        .withColumn(
            "transactions_per_customer",
            F.col("transaction_count") / F.col("unique_customers"),
        )
        .withColumn(
            "active_month_ratio",
            F.col("active_months") / F.lit(months_in_window),
        )
    )

    per_cust = (
        tx.groupBy("merchant_abn", "user_id")
        .agg(
            F.sum("dollar_value").alias("cust_gmv"),
            F.count("*").alias("cust_orders"),
        )
    )

    conc = (
        per_cust.groupBy("merchant_abn")
        .agg(
            (
                F.sum(F.when(F.col("cust_orders") > 1, 1).otherwise(0))
                / F.count("*")
            ).alias("repeat_rate"),
            F.sum(F.pow("cust_gmv", 2)).alias("_sq"),
            F.sum("cust_gmv").alias("_tot"),
        )
        .withColumn("customer_hhi", F.col("_sq") / F.pow(F.col("_tot"), 2))
        .drop("_sq", "_tot")
    )

    monthly = (
        tx.withColumn("month", F.trunc("order_datetime", "month"))
        .groupBy("merchant_abn", "month")
        .agg(F.sum("dollar_value").alias("monthly_gmv"))
    )

    month_starts = [
        (d.date(),)
        for d in pd.date_range(
            start=pd.Timestamp(window_start).replace(day=1),
            end=pd.Timestamp(window_end).replace(day=1),
            freq="MS",
        )
    ]
    complete_months = spark.createDataFrame(month_starts, ["month"])
    merchant_ids = tx.select("merchant_abn").distinct()

    monthly_complete = (
        merchant_ids.crossJoin(complete_months)
        .join(monthly, ["merchant_abn", "month"], "left")
        .fillna({"monthly_gmv": 0.0})
    )

    stab = (
        monthly_complete.groupBy("merchant_abn")
        .agg(
            F.avg("monthly_gmv").alias("mean_monthly_value"),
            F.stddev("monthly_gmv").alias("sd_monthly_value"),
        )
        .withColumn(
            "value_cv",
            F.when(
                F.col("mean_monthly_value") > 0,
                F.col("sd_monthly_value") / F.col("mean_monthly_value"),
            ),
        )
    )

    def three_month_block(start, end_exclusive, prefix):
        return (
            tx.filter(
                (F.col("order_datetime") >= F.lit(start))
                & (F.col("order_datetime") < F.lit(end_exclusive))
            )
            .groupBy("merchant_abn")
            .agg(
                F.sum("dollar_value").alias(f"{prefix}_3m_value"),
                F.countDistinct(F.trunc("order_datetime", "month")).alias(
                    f"{prefix}_active_months"
                ),
            )
        )

    early = three_month_block(early_start, early_end_exclusive, "early")
    recent = three_month_block(recent_start, recent_end_exclusive, "recent")

    growth = (
        early.join(recent, "merchant_abn", "full")
        .fillna(
            {
                "early_3m_value": 0.0,
                "recent_3m_value": 0.0,
                "early_active_months": 0,
                "recent_active_months": 0,
            }
        )
        .withColumn(
            "log_value_growth",
            F.when(
                (F.col("early_active_months") >= 2)
                & (F.col("recent_active_months") >= 2),
                F.log1p("recent_3m_value") - F.log1p("early_3m_value"),
            ),
        )
    )

    cp = per_cust.join(
        consumers.select(
            "user_id",
            "state",
            "sa2_code",
            "sa2_share",
            "abs_median_household_income_weekly",
            "seifa_irsad_decile",
        ),
        on="user_id",
        how="left",
    )

    has_income = F.col("abs_median_household_income_weekly").isNotNull()
    has_irsad = F.col("seifa_irsad_decile").isNotNull()
    wt = F.coalesce(F.col("sa2_share"), F.lit(1.0))

    base = (
        cp.groupBy("merchant_abn")
        .agg(
            (
                F.sum(
                    F.when(
                        has_income,
                        F.col("abs_median_household_income_weekly") * wt,
                    )
                )
                / F.sum(F.when(has_income, wt))
            ).alias("customer_income"),
            (
                F.sum(F.when(has_irsad, F.col("seifa_irsad_decile") * wt))
                / F.sum(F.when(has_irsad, wt))
            ).alias("customer_irsad"),
            F.avg(F.when(has_income, 1.0).otherwise(0.0)).alias(
                "pct_customers_with_abs"
            ),
        )
    )

    return (
        core.join(conc, "merchant_abn", "left")
        .join(stab.select("merchant_abn", "value_cv"), "merchant_abn", "left")
        .join(growth.select("merchant_abn", "log_value_growth"), "merchant_abn", "left")
        .join(base, "merchant_abn", "left")
        .join(
            merchants.select("merchant_abn", "name", "take_rate"),
            "merchant_abn",
            "left",
        )
        .withColumn(
            "estimated_bnpl_revenue",
            F.col("gmv") * F.col("take_rate") / 100.0,
        )
    )



def window_features(tx, merchants, consumers, window_start, window_end,
                    months_in_window=None, early_start=None, early_end_exclusive=None,
                    recent_start=None, recent_end_exclusive=None):
    start, end = pd.Timestamp(window_start), pd.Timestamp(window_end)
    if start.day != 1 or end != end + pd.offsets.MonthEnd(0) or end < start:
        raise ValueError("Feature windows must contain complete calendar months")
    months = (end.year - start.year) * 12 + end.month - start.month + 1
    if months_in_window is not None and months_in_window != months:
        raise ValueError("Month count must match the complete calendar window")
    return _build_window_features(
        tx, months, early_start or str(start.date()),
        early_end_exclusive or str((start + pd.DateOffset(months=3)).date()),
        recent_start or str((end.replace(day=1) - pd.DateOffset(months=2)).date()),
        recent_end_exclusive or str((end.replace(day=1) + pd.DateOffset(months=1)).date()),
        str(start.date()), str(end.date()), merchants=merchants, consumers=consumers,
    )


def build_features(tx, months_in_window, early_start, early_end_exclusive,
                   recent_start, recent_end_exclusive):
    return window_features(tx, merchants, consumers, TRAIN_START, TRAIN_END,
                           months_in_window, early_start, early_end_exclusive,
                           recent_start, recent_end_exclusive)


train_features = (
    build_features(
        train,
        months_in_window=12,
        early_start="2021-03-01",
        early_end_exclusive="2021-06-01",
        recent_start="2021-12-01",
        recent_end_exclusive="2022-03-01",
    )
    .filter(F.col("take_rate").isNotNull())
)

train_pd = train_features.toPandas()

assert train_pd["active_month_ratio"].le(1.0 + 1e-12).all(), (
    "active_month_ratio exceeded 1; check the training-window month count"
)

print(f"{len(train_pd):,} merchants rankable on the 12-month training window")
train_pd.head(3)


4,019 merchants rankable on the 12-month training window


,merchant_abn,gmv,transaction_count,unique_customers,active_months,transactions_per_customer,active_month_ratio,repeat_rate,customer_hhi,value_cv,log_value_growth,customer_income,customer_irsad,pct_customers_with_abs,name,take_rate,estimated_bnpl_revenue
0,83412691377,287032.970292,8183,6961,12,1.175549,1.0,0.160322,0.000222,0.217225,0.172766,1588.240332,4.689578,0.835081,Suspendisse Sagittis Nullam Associates,2.94,8438.769327
1,73256306726,871636.860927,3059,2873,12,1.064741,1.0,0.062304,0.000587,0.241786,0.050678,1592.217233,4.704765,0.825966,Id LLP,4.81,41925.733011
2,15613631617,323915.139007,1081,1057,12,1.022706,1.0,0.022706,0.001357,0.210218,-0.059460,1576.461183,4.634186,0.836329,Ante Industries,0.35,1133.702987


## 2. Score v1 exactly as it was originally defined

This section intentionally uses the original v1 weights and dollar-space shrinkage. The
March–August period is a **development window**, so it is valid to inspect the result here and
use the diagnosis to redesign hybrid. It is not valid to call the same period an untouched final test
after those changes have been made.


In [3]:
K = 300
WEIGHTS = {
    "bnpl_revenue_shrunk":       0.30,
    "growth_shrunk":             0.20,
    "transactions_per_customer": 0.10,
    "stability":                 0.10,
    "unique_customers":          0.05,
    "active_month_ratio":        0.05,
    "repeat_rate":               0.05,
    "customer_income":           0.05,
    "customer_irsad":            0.05,
    "customer_spread":           0.05,
}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9


def score(df):
    d = df.copy()
    d["shrinkage"] = d["transaction_count"] / (d["transaction_count"] + K)
    for raw, shrunk in [("estimated_bnpl_revenue", "bnpl_revenue_shrunk"),
                        ("log_value_growth", "growth_shrunk")]:
        d[shrunk] = d["shrinkage"] * d[raw] + (1 - d["shrinkage"]) * d[raw].mean()
    d["stability"] = -d["value_cv"]
    d["customer_spread"] = 1 - d["customer_hhi"]
    ranks = pd.DataFrame(
        {c: d[c].rank(pct=True, na_option="keep") * 100 for c in WEIGHTS}
    ).fillna(50.0)
    d["score"] = sum(ranks[c] * w for c, w in WEIGHTS.items())
    return d


train_scored = score(train_pd)
train_scored["predicted_rank"] = train_scored["score"].rank(ascending=False,
                                                            method="first").astype(int)
train_scored[["predicted_rank", "name", "estimated_bnpl_revenue", "score"]] \
    .sort_values("predicted_rank").head(10).round(2)

,predicted_rank,name,estimated_bnpl_revenue,score
3019,1,Odio Phasellus Institute,327603.13,88.06
2451,2,Quam Elementum Corp.,70718.65,87.73
2109,3,Lorem Ipsum Foundation,42920.10,86.83
478,4,Volutpat Nulla Incorporated,46083.78,86.53
2373,5,Tempus Non Lacinia Corporation,226412.68,86.10
3488,6,Dolor Quisque Inc.,175022.50,86.06
1153,7,Nec Ante Ltd,48983.36,85.91
2176,8,Donec Tempor Corp.,45884.06,85.76
1174,9,Consectetuer Cursus Inc.,41177.40,85.64
2867,10,Iaculis Aliquet Diam LLC,97236.46,85.61


## 3. What happened in the development window

Realised revenue to the firm over March–August 2022 is `development GMV × take_rate / 100`.

Merchants with no activity in this period are kept with realised revenue of zero. Excluding them
would make the ranking appear better by removing a genuine onboarding failure mode.


In [4]:
realised = (
    dev.groupBy("merchant_abn")
        .agg(
            F.sum("dollar_value").alias("dev_gmv"),
            F.count("*").alias("dev_transactions"),
        )
        .join(merchants.select("merchant_abn", "take_rate"), "merchant_abn", "left")
        .withColumn(
            "realised_revenue",
            F.col("dev_gmv") * F.col("take_rate") / 100.0,
        )
        .select(
            "merchant_abn",
            "dev_gmv",
            "dev_transactions",
            "realised_revenue",
        )
        .toPandas()
)

ev = train_scored.merge(realised, on="merchant_abn", how="left")
ev[["dev_gmv", "dev_transactions", "realised_revenue"]] = (
    ev[["dev_gmv", "dev_transactions", "realised_revenue"]].fillna(0.0)
)
ev["realised_rank"] = (
    ev["realised_revenue"].rank(ascending=False, method="first").astype(int)
)

print(f"merchants scored on training     : {len(ev):,}")
print(f"still trading in development     : {(ev['dev_transactions'] > 0).sum():,}")
print(f"stopped in development window    : {(ev['dev_transactions'] == 0).sum():,}")
print(
    f"total realised development revenue: "
    f"${ev['realised_revenue'].sum()/1e6:,.1f}M over 6 months"
)


merchants scored on training     : 4,019
still trading in development     : 3,991
stopped in development window    : 28
total realised development revenue: $30.3M over 6 months


## 4. Development diagnostics

**Spearman correlation** measures whether the score orders merchants similarly to their realised
development-window revenue.

**Capture rate** asks: if we had selected the predicted top 100, what share of the revenue earned
by the best possible 100 merchants in March–August would we have captured?

These are development metrics. They are useful for model design, but the final hybrid claim must come
from the September evaluation.


In [5]:
from scipy.stats import spearmanr

ours = ev.nsmallest(100, "predicted_rank")
baseline = ev.nlargest(100, "estimated_bnpl_revenue")
perfect = ev.nlargest(100, "realised_revenue")

rho_score, p_score = spearmanr(
    ev["score"],
    ev["realised_revenue"]
)

rho_base, p_base = spearmanr(
    ev["estimated_bnpl_revenue"],
    ev["realised_revenue"]
)

best_possible = perfect["realised_revenue"].sum()

ranking_rows = []

for label, sel in [
    ("our score", ours),
    ("baseline (past revenue)", baseline),
    ("perfect foresight", perfect),
]:
    ranking_rows.append({
        "picker": label,
        "hits in realised top 100": len(
            set(sel["merchant_abn"])
            & set(perfect["merchant_abn"])
        ),
        "realised revenue ($M)": (
            sel["realised_revenue"].sum() / 1e6
        ),
        "capture rate": (
            sel["realised_revenue"].sum()
            / best_possible
        ),
        "picks that stopped trading": int(
            (sel["dev_transactions"] == 0).sum()
        ),
    })

ranking_summary = (
    pd.DataFrame(ranking_rows)
    .set_index("picker")
)

print(
    f"Spearman, our score vs realised revenue : "
    f"{rho_score:.3f}  (p={p_score:.1e})"
)

print(
    f"Spearman, baseline   vs realised revenue: "
    f"{rho_base:.3f}  (p={p_base:.1e})"
)

print()

ranking_summary.round(3)

Spearman, our score vs realised revenue : 0.723  (p=0.0e+00)
Spearman, baseline   vs realised revenue: 0.977  (p=0.0e+00)



,hits in realised top 100,realised revenue ($M),capture rate,picks that stopped trading
picker,,,,
our score,42,6.307,0.598,0
baseline (past revenue),99,10.544,0.999,0
perfect foresight,100,10.551,1.000,0


In [6]:
overlap_ours_base = len(
    set(ours["merchant_abn"])
    & set(baseline["merchant_abn"])
)

print(
    f"our top 100 vs baseline top 100 overlap: "
    f"{overlap_ours_base}/100"
)

added = ours[
    ~ours["merchant_abn"].isin(
        baseline["merchant_abn"]
    )
]

dropped = baseline[
    ~baseline["merchant_abn"].isin(
        ours["merchant_abn"]
    )
]

print(
    f"\nmedian realised revenue of the "
    f"{len(added)} merchants we added   : "
    f"${added['realised_revenue'].median():,.0f}"
)

print(
    f"median realised revenue of the "
    f"{len(dropped)} merchants we dropped : "
    f"${dropped['realised_revenue'].median():,.0f}"
)

print(
    "\n(if the added merchants earn more than the dropped ones, "
    "the extra features paid off)"
)

our top 100 vs baseline top 100 overlap: 42/100

median realised revenue of the 58 merchants we added   : $21,575
median realised revenue of the 58 merchants we dropped : $87,121

(if the added merchants earn more than the dropped ones, the extra features paid off)


## 5. Which features helped in development?

The v1 score is a blend, so this section diagnoses which ingredients were associated with realised
March–August revenue. These correlations and candidate weightings are **model-development evidence**.
They may be used to choose hybrid, but they are not final out-of-sample performance estimates.


In [7]:
feature_cols = [
    "estimated_bnpl_revenue",
    "gmv",
    "transaction_count",
    "unique_customers",
    "transactions_per_customer",
    "repeat_rate",
    "value_cv",
    "active_month_ratio",
    "log_value_growth",
    "customer_income",
    "customer_irsad",
    "customer_hhi",
    "take_rate",
]

feature_rows = []

for c in feature_cols:

    s = ev[c].fillna(
        ev[c].median()
    )

    feature_rows.append({
        "feature": c,
        "spearman vs realised revenue": spearmanr(
            s,
            ev["realised_revenue"],
        )[0],
    })

feature_summary = (
    pd.DataFrame(feature_rows)
    .set_index("feature")
    .sort_values(
        "spearman vs realised revenue",
        key=abs,
        ascending=False,
    )
)

feature_summary.round(3)

,spearman vs realised revenue
feature,
estimated_bnpl_revenue,0.977
gmv,0.922
customer_hhi,-0.727
unique_customers,0.721
transaction_count,0.721
transactions_per_customer,0.681
repeat_rate,0.681
value_cv,-0.661
active_month_ratio,0.438


In [8]:
ranks_ev = pd.DataFrame(
    {
        c: (
            ev[c]
            .rank(
                pct=True,
                na_option="keep",
            )
            * 100
        )
        for c in WEIGHTS
    }
).fillna(50.0)

best_possible = perfect[
    "realised_revenue"
].sum()


def try_weights(label, weights):

    total = sum(
        weights.values()
    )

    s = sum(
        ranks_ev[c] * (w / total)
        for c, w in weights.items()
    )

    sel = (
        ev.assign(_s=s)
        .nlargest(100, "_s")
    )

    return {
        "weighting": label,

        "spearman": spearmanr(
            s,
            ev["realised_revenue"],
        )[0],

        "capture rate": (
            sel["realised_revenue"].sum()
            / best_possible
        ),

        "hits in realised top 100": len(
            set(sel["merchant_abn"])
            & set(perfect["merchant_abn"])
        ),
    }


variants = [
    try_weights(
        "current score (10 features)",
        WEIGHTS,
    ),

    try_weights(
        "revenue only (shrunk)",
        {
            "bnpl_revenue_shrunk": 1.0
        },
    ),

    try_weights(
        "revenue 70 / growth 30",
        {
            "bnpl_revenue_shrunk": 0.7,
            "growth_shrunk": 0.3,
        },
    ),

    try_weights(
        "revenue 60 / growth 20 / engagement 20",
        {
            "bnpl_revenue_shrunk": 0.6,
            "growth_shrunk": 0.2,
            "transactions_per_customer": 0.1,
            "repeat_rate": 0.1,
        },
    ),

    try_weights(
        "current score, no ABS features",
        {
            k: v
            for k, v in WEIGHTS.items()
            if k not in (
                "customer_income",
                "customer_irsad",
            )
        },
    ),
]


# Plain past-revenue baseline,
# without shrinkage.
plain = (
    ev["estimated_bnpl_revenue"]
    .rank(pct=True)
    * 100
)

sel = (
    ev.assign(_s=plain)
    .nlargest(100, "_s")
)

variants.append({
    "weighting": "past revenue, no shrinkage",

    "spearman": spearmanr(
        plain,
        ev["realised_revenue"],
    )[0],

    "capture rate": (
        sel["realised_revenue"].sum()
        / best_possible
    ),

    "hits in realised top 100": len(
        set(sel["merchant_abn"])
        & set(perfect["merchant_abn"])
    ),
})

variant_summary = (
    pd.DataFrame(variants)
    .set_index("weighting")
)

variant_summary.round(3)

,spearman,capture rate,hits in realised top 100
weighting,,,
current score (10 features),0.723,0.598,42
revenue only (shrunk),0.487,0.999,99
revenue 70 / growth 30,0.448,0.417,25
revenue 60 / growth 20 / engagement 20,0.543,0.677,50
"current score, no ABS features",0.736,0.614,44
"past revenue, no shrinkage",0.977,0.999,99


## 6. Development-window pictures

In [9]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE = "#4C78A8"
ORANGE = "#F58518"
GREY = "#8C8C8C"

fig, ax = plt.subplots(
    figsize=(6.4, 6)
)

ax.scatter(
    ev["predicted_rank"],
    ev["realised_rank"],
    s=6,
    alpha=0.25,
    color=GREY,
    edgecolors="none",
)

ax.scatter(
    ours["predicted_rank"],
    ours["realised_rank"],
    s=14,
    color=ORANGE,
    edgecolors="none",
    label="our predicted top 100",
)

lim = len(ev)

ax.plot(
    [1, lim],
    [1, lim],
    color=BLUE,
    linewidth=1.5,
    label="perfect prediction",
)

ax.set_xlim(0, lim)
ax.set_ylim(0, lim)

ax.invert_xaxis()
ax.invert_yaxis()

ax.set_xlabel(
    "predicted rank (train window)"
)

ax.set_ylabel(
    "realised rank (development window)"
)

ax.set_title(
    f"Training score vs development revenue rank - "
    f"Spearman {rho_score:.2f}"
)

ax.legend(
    loc="lower left",
    frameon=False,
)

plt.tight_layout()
plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27071/153272034.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
caps = (
    ranking_summary[
        "capture rate"
    ]
    * 100
)

fig, ax = plt.subplots(
    figsize=(7, 2.8)
)

colors = [
    ORANGE,
    GREY,
    BLUE,
]

bars = ax.barh(
    caps.index[::-1],
    caps.values[::-1],
    color=colors[::-1],
    height=0.6,
)

ax.bar_label(
    bars,
    labels=[
        f"{v:.0f}%"
        for v in caps.values[::-1]
    ],
    padding=4,
    fontsize=9,
)

ax.set_xlim(
    0,
    105,
)

ax.set_xlabel(
    "% of the realised revenue a perfect "
    "top 100 would have earned"
)

ax.set_title(
    "How much of the achievable revenue "
    "each picker captured"
)

plt.tight_layout()
plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27071/3050612575.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Reading this result correctly

This notebook is the **v1 post-mortem**. Use the tables above to explain what the development
window taught the group — especially whether the ten-feature score beat the much simpler past-
revenue baseline, which features carried predictive signal, and what the weight sweep suggested.

The important methodological point is what happens next:

1. March–August 2022 has now been inspected, so it is part of model development.
2. The hybrid architecture and weights may be chosen using this evidence.
3. Once hybrid is frozen, evaluate it **once** on September 2022 in
   `merchant_ranking_hybrid.ipynb`.
4. Do not change hybrid after seeing the September result. If a later change is made, September
   becomes development data too and a new September evaluation would be required.

Limitations remain: one six-month development window is not repeated cross-validation, and the
synthetic data may have snapshot-specific structure. The September holdout is also only one month,
so it is a cleaner test of leakage, not a claim of long-run certainty.
